In [ ]:
!pip3 install timm

  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 44.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.3/564.3 kB 20.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.4 MB/s  0:00:00
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [timm]4/5 [timm]ngface_hub]


In [ ]:
# ==============================================================================
# PART 1: HYBRID FEATURE EXTRACTION (GPU REQUIRED)
# Switching to stable LeViT-256 to fix positional embedding mismatch
# ==============================================================================

import os
import time
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import timm
import numpy as np
from tqdm import tqdm

# --- Configuration (UPDATED) ---
DATA_ROOT = "/workspace/"
TRAIN_PATH = os.path.join(DATA_ROOT, "Train")
VALID_PATH = os.path.join(DATA_ROOT, "Valid")
TEST_PATH = os.path.join(DATA_ROOT, "Test")

BATCH_SIZE = 64
NUM_WORKERS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# *** CRITICAL CHANGE: SWITCH TO A STABLE IMAGE SIZE ***
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = 224 # <--- Changed from 384 to 256 for stable LeViT-256 weights

# --- Data Transformation ---
necessary_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), # Now 256x256
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD)
])

# --- 1. Load Datasets Directly from Separate Folders ---
print(f"Loading data from: {DATA_ROOT}")
train_set = datasets.ImageFolder(TRAIN_PATH, transform=necessary_transform)
val_set = datasets.ImageFolder(VALID_PATH, transform=necessary_transform)
test_set = datasets.ImageFolder(TEST_PATH, transform=necessary_transform)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"Train samples: {len(train_set)}, Valid samples: {len(val_set)}, Test samples: {len(test_set)}")


# --- 2. Hybrid Feature Extractor Model (FINAL FIX) ---
class HybridFeatureExtractor(nn.Module):
    """Fuses features from LeViT and EfficientNetV2."""
    def __init__(self):
        super().__init__()

        # LeViT branch (Vision Transformer) - USING STABLE LEVIT-256
        self.levit = timm.create_model(
            'levit_128', # <--- Changed model name
            pretrained=True,
            num_classes=0,
            img_size=IMAGE_SIZE # Ensures model configuration matches input size
        )

        # EfficientNetV2-S branch (CNN) - Stable name kept
        self.efficientnet = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=0)

    def forward(self, x):
        levit_features = self.levit(x)
        efficient_features = self.efficientnet(x)
        # Feature Fusion
        combined_features = torch.cat((levit_features, efficient_features), dim=1)
        return combined_features

# --- 3. Feature Extraction Function ---
def extract_features(data_loader, split_name):
    """Extracts features and records time."""
    start_time = time.time()
    model.eval()
    all_features = []
    all_labels = []

    print(f"Starting feature extraction for {split_name}...")
    with torch.no_grad():
        for images, labels in tqdm(data_loader):
            images = images.to(DEVICE)
            features = model(images).cpu().numpy()
            all_features.append(features)
            all_labels.extend(labels.tolist())

    features_matrix = np.concatenate(all_features, axis=0)
    labels_array = np.array(all_labels)
    elapsed_time = time.time() - start_time

    return features_matrix, labels_array, elapsed_time

# Instantiate Model
model = HybridFeatureExtractor().to(DEVICE)

# Run Extraction for all splits
X_train, y_train, train_time_feat = extract_features(train_loader, "Training")
X_val, y_val, val_time_feat = extract_features(val_loader, "Validation")
X_test, y_test, test_time_feat = extract_features(test_loader, "Testing")

print(f"\nFeature Extraction Times:")
print(f"Train Feat Time: {train_time_feat:.2f}s | Val Feat Time: {val_time_feat:.2f}s | Test Feat Time: {test_time_feat:.2f}s")

# Save Extracted Features
np.save(os.path.join(DATA_ROOT, 'X_train.npy'), X_train)
np.save(os.path.join(DATA_ROOT, 'y_train.npy'), y_train)
np.save(os.path.join(DATA_ROOT, 'X_valid.npy'), X_val)
np.save(os.path.join(DATA_ROOT, 'y_valid.npy'), y_val)
np.save(os.path.join(DATA_ROOT, 'X_test.npy'), X_test)
np.save(os.path.join(DATA_ROOT, 'y_test.npy'), y_test)

Using device: cuda
Loading data from: /workspace/
Train samples: 18898, Valid samples: 2362, Test samples: 2364


model.safetensors:   0%|          | 0.00/37.1M [00:00<?, ?B/s]

Unexpected keys (head.bn.num_batches_tracked, head.bn.bias, head.bn.running_mean, head.bn.running_var, head.bn.weight, head_dist.bn.num_batches_tracked, head_dist.bn.bias, head_dist.bn.running_mean, head_dist.bn.running_var, head_dist.bn.weight) found while loading pretrained weights. This may be expected if model is being adapted.


Starting feature extraction for Training...


  0%|          | 0/296 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  2%|▏         | 6/296 [00:01<00:55,  5.23it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
  4%|▍         | 12/296 [00:02<00:39,  7.19it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 10%|█         | 30/296 [00:06<00:57,  4.60it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 296/296 [01:00<00:00,  4.93it/s]


Starting feature extraction for Validation...


  0%|          | 0/37 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 11%|█         | 4/37 [00:01<00:07,  4.37it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 57%|█████▋    | 21/37 [00:06<00:05,  3.18it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 37/37 [00:09<00:00,  3.75it/s]


Starting feature extraction for Testing...


  0%|          | 0/37 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 14%|█▎        | 5/37 [00:01<00:07,  4.15it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
 41%|████      | 15/37 [00:03<00:05,  4.24it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1054: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
100%|██████████| 37/37 [00:08<00:00,  4.59it/s]



Feature Extraction Times:
Train Feat Time: 60.53s | Val Feat Time: 10.59s | Test Feat Time: 8.09s


In [ ]:
!pip3 install pandas


  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.4 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
Using cached tzdata-2025.2-py2.py3-none-any.whl (347 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]2m2/3 [pandas]


In [ ]:
# ==============================================================================
# PART 2: XGBOOST METRICS RE-CALCULATION (MODIFIED FOR LOG LOSS)
# ==============================================================================

import time
# Re-import all necessary modules, including log_loss
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, precision_recall_fscore_support,
    log_loss # <--- CRITICAL: New Import
)
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore', category=UserWarning) # Suppress the XGBoost warning if re-running fit

# Assuming all previous variables (X_train, y_train, xgb_model, feature times) are in memory.

# --- 3. Define Metric Calculation Function (MODIFIED) ---
def compute_metrics(X, y_true, split_name, model, is_train=False, feature_time=0):
    """Computes all requested metrics for a given split, including Log Loss."""

    # 1. Prediction and Timing
    start_time_pred = time.time()
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)
    time_taken_pred = time.time() - start_time_pred

    # 2. Core Metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )

    # 3. TPR and FPR (Macro Avg) - standard calculation
    cm = confusion_matrix(y_true, y_pred)
    TP = np.diag(cm)
    FP = cm.sum(axis=0) - TP
    FN = cm.sum(axis=1) - TP
    TN = cm.sum() - (FP + FN + TP)

    epsilon = 1e-7
    TPR = np.mean(TP / (TP + FN + epsilon))
    FPR = np.mean(FP / (FP + TN + epsilon))

    # 4. AUROC and LOSS Calculation
    try:
        auroc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
    except ValueError:
        auroc = np.nan

    current_loss = 'N/A (XGBoost Objective)' # Default for training
    if not is_train:
        # Calculate Log Loss (Cross-Entropy Loss) for validation and test sets
        try:
            current_loss = log_loss(y_true, y_proba)
        except ValueError:
            current_loss = np.nan

    metrics = {
        f'{split_name} Accuracy': accuracy,
        f'{split_name} Precision': precision,
        f'{split_name} Recall': recall,
        f'{split_name} F1 Score': f1,
        f'{split_name} TPR (Macro Avg)': TPR,
        f'{split_name} FPR (Macro Avg)': FPR,
        f'{split_name} Loss': current_loss, # <--- UPDATED
        f'{split_name} Time (s)': time_taken_pred + feature_time,
    }

    if not is_train:
        metrics[f'{split_name} AUROC (Macro Avg)'] = auroc

    return metrics, classification_report(y_true, y_pred)


# --- 4. Compute Metrics for All Splits ---

# Assuming X_train, y_train, xgb_model, and all time variables are defined.

# Training Metrics
# Note: Training loss remains 'N/A' as we don't calculate it post-hoc on the training set
train_metrics, train_report = compute_metrics(
    X_train, y_train, "Training", xgb_model, is_train=True, feature_time=train_time_feat
)
train_metrics['Training Time (s)'] = train_time_xgb + train_time_feat

# Validation Metrics
val_metrics, val_report = compute_metrics(
    X_val, y_val, "Validation", xgb_model, feature_time=val_time_feat
)

# Testing Metrics
test_metrics, test_report = compute_metrics(
    X_test, y_test, "Testing", xgb_model, feature_time=test_time_feat
)

# --- 5. Print Results ---
print("\n" + "="*80)
print("FINAL HYBRID MODEL METRICS SUMMARY (LeViT + EfficientNetV2-S + XGBoost)")
print("="*80)

all_metrics = {**train_metrics, **val_metrics, **test_metrics}
df_metrics = pd.DataFrame(
    list(all_metrics.items()), columns=['Metric', 'Value']
).set_index('Metric')
print(df_metrics)
# Classification reports remain unchanged


FINAL HYBRID MODEL METRICS SUMMARY (LeViT + EfficientNetV2-S + XGBoost)
                                                Value
Metric                                               
Training Accuracy                            0.999418
Training Precision                           0.999422
Training Recall                              0.999427
Training F1 Score                            0.999425
Training TPR (Macro Avg)                     0.999427
Training FPR (Macro Avg)                     0.000073
Training Loss                 N/A (XGBoost Objective)
Training Time (s)                          132.745068
Validation Accuracy                          0.859018
Validation Precision                         0.856827
Validation Recall                            0.855301
Validation F1 Score                          0.855688
Validation TPR (Macro Avg)                   0.855301
Validation FPR (Macro Avg)                   0.017627
Validation Loss                              0.481264
Validatio